In [0]:
# df = pd.read_csv("orders_hw41.csv", parse_dates=["placed_at"])
# is_paid = df[df["status"] == "paid"]
# result = is_paid.groupby("country").agg({"price_each": "sum", "id": "count"}).rename(columns={"price_each": "revenue", "id": "orders"})

# report.py — given
import pandas as pd
df = pd.read_csv("orders_hw41.csv", parse_dates=["placed_at"])
df["revenue"] = df["quantity"] * df["price_each"]
df["month"] = df["placed_at"].dt.to_period("M").astype(str)
paid = df[df["status"] == "paid"]
monthly = (paid.groupby(["country", "month"], as_index=False)
 .agg(revenue=("revenue", "sum"), orders=
("order_id", "count")))
top = (monthly.sort_values(["country", "revenue"], ascending=
[True, False])
 .groupby("country", as_index=False).head(1))
top.to_csv("best_month_per_country.csv", index=False)
print(top)

In [0]:
import polars as pl
df = pl.read_csv("orders_hw41.csv")

df = df.with_columns([
    (pl.col("quantity") * pl.col("price_each")).alias("revenue"),

    pl.col("placed_at")
      .str.strptime(pl.Date, "%Y-%m-%d", strict=False)
      .dt.strftime("%Y-%m")
      .alias("month")
])

paid = (df.filter(pl.col("status") == "paid"))

monthly = (paid.group_by("country", "month")).agg((pl.col("revenue").sum()).alias("revenue"), pl.col("order_id").count().alias("orders"))

top = (monthly.sort("country", "revenue", descending=[False, True])
 .group_by("country")
 .head(1)
 .sort("country"))

top.write_csv("best_month_per_country_polars.csv")

print(top)